<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_TMS_fMRI_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_TMS_fMRI_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulate TMS-fMRI Sessions with Subject-Specific, Stimulus-Agnostic ANN Models

**Objective:** Use subject-specific models trained on TMS data (without explicit stimulus encoding) to simulate both resting-state and stimulation sessions with empirical TMS timing and target location information.

**Key Innovation:** Unlike stimulus-aware models, these models capture intrinsic brain dynamics learned from recordings during stimulation. When we apply realistic TMS timing and spatial location during simulation, we test whether learned dynamics naturally respond to stimulation constraints.

**Workflow:**
1. Load subject-specific stimulus-agnostic MLP models trained on concatenated TMS sessions
2. Load empirical dataset with rest and stim sessions, including TMS target regions and timing
3. For each subject, generate synthetic sessions:
   - Rest: Simulate resting-state activity without stimulation
   - Stim: Simulate brain response to empirical TMS timing and target location using spatial Gaussian kernel
4. Save simulated dataset to preprocessed_subjects_tms_fmri
5. Validate simulations match empirical functional connectivity

## Step 1: Setup and Configuration

In [ ]:
# --- Setup ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, pickle, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Clone repo + add to path
repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
if not os.path.exists(repo_dir):
    !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
else:
    print("Repo already exists ✅")

sys.path.append(repo_dir)
from src.preprocessing_tms_fmri import preprocess_run
from src.NPI import build_model, device

print(f"PyTorch device: {device}")

In [ ]:
# --- Configuration Parameters ---
ROI_num = 450
using_steps = 3
tr_rest = 2.0
tr_stim = 2.4
remove_initial_trs = 12  # Matching new model training (not 30)
low_hz, high_hz = 0.008, 0.08

# Simulation parameters (NEW VALUES)
BURN_IN = 30
NOISE_SIGMA = 0.28
STIM_AMP = 10.0
STIM_DURATION_S = tr_stim
RHO_MM = 60.0  # Gaussian spread (mm) for spatial TMS effect

rng = np.random.default_rng(42)

# Paths
base_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
data_dir = os.path.join(base_dir, "TMS_fMRI")
preproc_dir = os.path.join(base_dir, "preprocessed_subjects_tms_fmri")
dataset_pkl = os.path.join(data_dir, "dataset_tian50_schaefer400_allruns.pkl")

# Load models from NEW training workflow
weights_dir = os.path.join(preproc_dir, "trained_models_MLP")  # New stimulus-agnostic models

# Output directory for simulated dataset
out_dir = os.path.join(preproc_dir, "simulated_tms_fmri_stim_info_v2")
os.makedirs(out_dir, exist_ok=True)

out_pkl = os.path.join(out_dir, "dataset_simulated_with_stim_info.pkl")
results_json = os.path.join(out_dir, "simulation_summary.json")

print(f"Base dir: {base_dir}")
print(f"Weights dir: {weights_dir}")
print(f"Output dir: {out_dir}")
print(f"\nConfiguration:")
print(f"  ROI_num: {ROI_num}, using_steps: {using_steps}")
print(f"  TR_rest: {tr_rest}s, TR_stim: {tr_stim}s")
print(f"  remove_initial_trs: {remove_initial_trs}")
print(f"  BURN_IN: {BURN_IN}, NOISE_SIGMA: {NOISE_SIGMA}, STIM_AMP: {STIM_AMP}, RHO_MM: {RHO_MM}mm")

## Step 2: Load Distance Matrix and Spatial Gaussian Kernel

In [ ]:
# --- Load distance matrix for spatial Gaussian kernel ---
try:
    dist_path = os.path.join(data_dir, "atlases", "distance_matrix_450x450_Tian50_Schaefer400.npy")
    D = np.load(dist_path)
    W = np.exp(-(D ** 2) / (2.0 * (RHO_MM ** 2))).astype(np.float32)
    W /= (W[np.arange(ROI_num), np.arange(ROI_num)][:, None] + 1e-8)  # Normalize so target = 1
    print(f"✓ Loaded distance matrix: W shape {W.shape}, range [{W.min():.4f}, {W.max():.4f}]")
    print(f"  Gaussian spread: RHO_MM = {RHO_MM} mm")
    print(f"  Diagonal values (self-weights): {W.diagonal()[:5]}...")
except FileNotFoundError as e:
    print(f"⚠ Distance matrix not found: {e}")
    print(f"  Will use direct stimulation only (no spatial spread)")
    W = None

## Step 3: Define Model Architecture and Load Trained Models

In [ ]:
# --- Model Architecture: Stimulus-Agnostic MLP ---
# Using NPI helper function instead of defining class manually
# Model architecture:
#   Input: Brain state only (using_steps * ROI_num dimensions = 3 * 450 = 1350)
#   Hidden layers: 256 → 128 hidden units with ReLU activations
#   Output: Next predicted ROI activation (450 dims)
# Key difference from old models: NO stimulus encoding. Just intrinsic dynamics.

print("✓ Will instantiate models using src.NPI.build_model()")

In [ ]:
# --- Load subject-specific trained models ---
print("Loading subject-specific stimulus-agnostic models...\n")

trained_models = {}
model_pattern = '_MLP.pt'  # New models save as sub-ID_MLP.pt

# List available model files
if os.path.exists(weights_dir):
    model_files = [f for f in os.listdir(weights_dir) if f.endswith(model_pattern)]
    print(f"Found {len(model_files)} model files in {weights_dir}")
else:
    print(f"⚠ Weights directory not found: {weights_dir}")
    model_files = []

for model_file in sorted(model_files):
    sub_id = model_file.replace(model_pattern, '')
    model_path = os.path.join(weights_dir, model_file)

    try:
        # Load the checkpoint
        torch.serialization.add_safe_globals([torch.nn.modules.linear.Linear, torch.nn.modules.activation.ReLU])
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)

        # Handle both cases: saved as state_dict or as full model
        if isinstance(checkpoint, dict):
            # Case 1: Saved as state_dict
            model = build_model("MLP", ROI_num=ROI_num, using_steps=using_steps).to(device)
            model.load_state_dict(checkpoint)
        else:
            # Case 2: Saved as full model object
            model = checkpoint.to(device) if hasattr(checkpoint, 'to') else checkpoint

        model.eval()
        trained_models[sub_id] = model
        print(f"✓ {sub_id}")
    except Exception as e:
        print(f"✗ {sub_id} - skipping ({str(e)[:50]}...)")

print(f"\n✅ Loaded {len(trained_models)} models ready for simulation")

## Step 4: Load Empirical Dataset and Verify Subject Matching

In [ ]:
# --- Load empirical dataset ---
print(f"Loading empirical dataset from {dataset_pkl}...")
with open(dataset_pkl, "rb") as f:
    dataset_emp = pickle.load(f)

print(f"✓ Loaded {len(dataset_emp)} subjects")

# Verify dataset structure
sample_sub = list(dataset_emp.keys())[0]
print(f"\nSample subject: {sample_sub}")
print(f"  Available conditions: {list(dataset_emp[sample_sub].keys())}")
if 'task-rest' in dataset_emp[sample_sub]:
    sample_rest = dataset_emp[sample_sub]['task-rest']
    print(f"  Rest runs: {list(sample_rest.keys())[:3]}...")
    first_rest_run = list(sample_rest.values())[0]
    print(f"  Rest data keys: {list(first_rest_run.keys())}")
    if 'time series' in first_rest_run:
        print(f"  Rest TS shape: {first_rest_run['time series'].shape}")

if 'task-stim' in dataset_emp[sample_sub]:
    sample_stim = dataset_emp[sample_sub]['task-stim']
    print(f"  Stim runs: {list(sample_stim.keys())[:3]}...")
    first_stim_run = list(sample_stim.values())[0]
    print(f"  Stim data keys: {list(first_stim_run.keys())}")
    if 'target' in first_stim_run:
        print(f"  Stim target: {first_stim_run['target']}")
    if 'stim time' in first_stim_run:
        print(f"  Stim time type: {type(first_stim_run['stim time'])}")

# Match models to empirical data
subjects_with_models_and_data = []
for sub_id in trained_models.keys():
    if sub_id in dataset_emp:
        subjects_with_models_and_data.append(sub_id)

print(f"\n{'='*70}")
print(f"Subjects with both trained models AND empirical data: {len(subjects_with_models_and_data)}")
print(f"  {subjects_with_models_and_data[:5]}..." if len(subjects_with_models_and_data) > 5 else f"  {subjects_with_models_and_data}")
print(f"{'='*70}")

## Step 5: Simulation Helper Functions

In [ ]:
# --- Helper Functions for Data Parsing ---

def safe_target_idx(target_vec):
    """Extract target region index from one-hot vector."""
    if target_vec is None:
        return None
    v = np.asarray(target_vec).astype(int).ravel()
    if v.size == 0 or v.sum() != 1:
        return None
    return int(np.argmax(v))

def get_onset_column(df):
    """Find onset column in dataframe."""
    if df is None or len(df) == 0:
        return None
    for col in ["onset", "Onset", "stim_onset", "onset_s", "onset_sec", "time", "t", "seconds"]:
        if col in df.columns:
            return col
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            return col
    return None

def map_onsets_to_steps(onsets_s, tr_model=tr_stim, mode="round"):
    """Map stimulus onsets (seconds) to model steps.

    After removing initial TRs, stimulus times are relative to start of cleaned data.
    This function converts seconds to simulation step indices.
    """
    onsets_s = np.asarray(onsets_s, dtype=float)
    x = onsets_s / float(tr_model)
    if mode == "round":
        steps = np.rint(x).astype(int)
    elif mode == "floor":
        steps = np.floor(x).astype(int)
    else:
        steps = np.ceil(x).astype(int)
    steps = steps[steps >= 0]
    return np.unique(steps)

print("✓ Helper functions defined")

In [ ]:
# --- Neural Network Inference ---

@torch.no_grad()
def predict_next(model, window_SxN):
    """Predict next state from rolling window of brain state.

    For stimulus-agnostic models:
    - Input: flattened brain state from using_steps (1350 dims for 3 steps × 450 ROIs)
    - Output: next predicted ROI activation (450 dims)
    - Noise is added to output for stochasticity

    Args:
        model: ANN_MLP model instance
        window_SxN: (using_steps, ROI_num) array of recent brain states

    Returns:
        (ROI_num,) predicted next state with noise
    """
    # Flatten brain state
    brain_state = window_SxN.reshape(-1).astype(np.float32)

    # Add input noise
    noise = NOISE_SIGMA * rng.normal(0.0, 1.0, size=brain_state.shape).astype(np.float32)
    brain_state = brain_state + noise

    # Forward pass
    x = torch.tensor(brain_state[None, :], dtype=torch.float32, device=device)
    y = model(x)

    return y.detach().cpu().numpy().squeeze(0)

def simulate_run(model, init_window_SxN, n_steps, stim_steps=None, target_idx=None, W=None, stim_amplitude=None):
    """Simulate brain activity time series with optional TMS stimulation.

    The model learns intrinsic dynamics during normal brain function.
    When we apply stimulation, we perturb the state at the target region using:
    - Spatial Gaussian kernel (W) to spread stimulation across regions
    - Empirical stimulus timing (stim_steps) to apply perturbations at correct times

    Args:
        model: Trained ANN_MLP
        init_window_SxN: (using_steps, ROI_num) initial brain state window
        n_steps: Number of simulation steps
        stim_steps: Set of step indices when stimulation occurs
        target_idx: Target ROI for stimulation (0-449)
        W: (ROI_num, ROI_num) spatial Gaussian kernel for TMS spread
        stim_amplitude: Stimulation amplitude. If None, uses global STIM_AMP. Set to 0.0 for control (no perturbation).

    Returns:
        (n_steps, ROI_num) simulated time series
    """
    init_window_SxN = np.asarray(init_window_SxN, dtype=np.float32)
    assert init_window_SxN.shape == (using_steps, ROI_num)

    # Use provided amplitude or default to global STIM_AMP
    if stim_amplitude is None:
        stim_amplitude = STIM_AMP

    # Prepare stimulation parameters
    if stim_steps is not None:
        stim_steps = set(int(s) for s in stim_steps)
    else:
        stim_steps = set()

    do_stim = (target_idx is not None) and (len(stim_steps) > 0)

    w = init_window_SxN.copy()

    # Burn-in phase: let model settle to attractor without stimulation
    for _ in range(BURN_IN):
        y = predict_next(model, w)
        w = np.vstack([w[1:], y[None, :]])

    # Simulate with optional spatial TMS
    out = np.zeros((n_steps, ROI_num), dtype=np.float32)
    for t in range(n_steps):
        # Apply stimulation if this is a stim step
        if do_stim and (t in stim_steps):
            if W is not None:
                # Apply spatial Gaussian spread of stimulation
                w[-1, :] += stim_amplitude * W[target_idx, :]
            else:
                # No spatial kernel - apply to target region only
                w[-1, target_idx] += stim_amplitude

        # Get model prediction
        y = predict_next(model, w)
        out[t] = y

        # Update rolling window
        w = np.vstack([w[1:], y[None, :]])

    return out

print("✓ Simulation functions defined")

## Step 6: Functional Connectivity Computation

In [ ]:
# --- Functional Connectivity Functions ---

def compute_fc_matrix(time_series):
    """Compute functional connectivity (Pearson correlation) matrix.

    Args:
        time_series: (T, ROI_num) time series array

    Returns:
        (ROI_num, ROI_num) correlation matrix
    """
    ts_std = (time_series - time_series.mean(axis=0)) / (time_series.std(axis=0) + 1e-8)
    fc = np.corrcoef(ts_std.T)
    return fc

def extract_upper_triangle(fc_matrix):
    """Extract upper triangle of FC matrix (excluding diagonal) as vector.

    For comparison, this removes redundant lower triangle and diagonal.
    """
    N = fc_matrix.shape[0]
    indices = np.triu_indices(N, k=1)
    return fc_matrix[indices]

print("✓ FC computation functions defined")

## Step 7: Generate Simulated Dataset Using Empirical TMS Parameters

In [ ]:
# --- Generate synthetic dataset ---
print("="*70)
print("GENERATING SIMULATED DATASET")
print("="*70)
print("\nFor each subject with trained model:")
print("  1. Simulate REST sessions (no stimulation)")
print("  2. Simulate STIM sessions (with empirical TMS timing & target)")
print()

dataset_sim = {}
n_sim_rest = 0
n_sim_stim = 0
sim_errors = []

for sub_id in sorted(subjects_with_models_and_data):
    model = trained_models[sub_id]
    sub_data_emp = dataset_emp[sub_id]

    dataset_sim[sub_id] = {"task-rest": {}, "task-stim": {}}

    # ---- SIMULATE REST ----
    if "task-rest" in sub_data_emp:
        rest_count = 0
        for run_idx, run_emp in sub_data_emp["task-rest"].items():
            ts_emp = run_emp.get("time series", None)

            # Validate data
            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            try:
                # Preprocess empirical rest (drop TRs, bandpass, z-score)
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                # Use first using_steps timesteps as initial window, no stimulation
                init_window = ts_proc[:using_steps].copy()
                n_steps = ts_proc.shape[0]

                # Simulate rest
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=None, target_idx=None, W=None)

                dataset_sim[sub_id]["task-rest"][int(run_idx)] = {
                    "time series": sim_ts,
                    "metadata": {"simulated": True, "tr": float(tr_rest), "n_vols": n_steps}
                }
                rest_count += 1
                n_sim_rest += 1
            except Exception as e:
                sim_errors.append(f"{sub_id} rest run {run_idx}: {str(e)[:60]}")

        if rest_count > 0:
            print(f"  {sub_id}: Simulated {rest_count} rest sessions")

    # ---- SIMULATE STIM ----
    if "task-stim" in sub_data_emp:
        stim_count = 0
        for run_idx, run_emp in sub_data_emp["task-stim"].items():
            ts_emp = run_emp.get("time series", None)
            target_vec = run_emp.get("target", None)
            stim_time_df = run_emp.get("stim time", None)

            # Validate data
            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            # Extract target region
            target_idx = safe_target_idx(target_vec)
            if target_idx is None:
                continue

            try:
                # Preprocess empirical stim (drop TRs, bandpass, z-score)
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                # Extract stimulus timing
                stim_steps_model = []
                if stim_time_df is not None and len(stim_time_df) > 0:
                    onset_col = get_onset_column(stim_time_df)
                    if onset_col is not None:
                        onsets_s = stim_time_df[onset_col].values
                        # Subtract initial TRs that were dropped (in seconds)
                        onsets_s = onsets_s[onsets_s >= remove_initial_trs * tr_stim]
                        onsets_s = onsets_s - remove_initial_trs * tr_stim
                        stim_steps_model = map_onsets_to_steps(onsets_s, tr_model=tr_stim)

                if len(stim_steps_model) == 0:
                    # No valid stimulation times
                    continue

                # Use first using_steps timesteps as initial window
                init_window = ts_proc[:using_steps].copy()
                n_steps = ts_proc.shape[0]

                # Simulate stim with empirical target and timing
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=stim_steps_model, target_idx=target_idx, W=W)

                dataset_sim[sub_id]["task-stim"][int(run_idx)] = {
                    "time series": sim_ts,
                    "target": target_vec,
                    "stim time": stim_time_df,
                    "metadata": {
                        "simulated": True,
                        "tr": float(tr_stim),
                        "target_idx": int(target_idx),
                        "n_stim_pulses": len(stim_steps_model),
                        "n_vols": n_steps
                    }
                }
                stim_count += 1
                n_sim_stim += 1
            except Exception as e:
                sim_errors.append(f"{sub_id} stim run {run_idx}: {str(e)[:60]}")

        if stim_count > 0:
            print(f"  {sub_id}: Simulated {stim_count} stim sessions")

print("\n" + "="*70)
print(f"✓ Generated {n_sim_rest} rest sessions and {n_sim_stim} stim sessions")
print(f"✓ Simulated {len(dataset_sim)} subjects total")

if sim_errors:
    print(f"\n⚠ Encountered {len(sim_errors)} errors during simulation:")
    for err in sim_errors[:3]:
        print(f"  {err}")
    if len(sim_errors) > 3:
        print(f"  ... and {len(sim_errors)-3} more")

## Step 7b: Generate Control Dataset - REST INIT for Both Conditions

**KEY INNOVATION**: Both REST and STIM (amplitude=0) use **REST empirical initialization**

This creates a **true amplitude-only comparison**:
- Same initial brain state (from REST)
- Different amplitude (100.0 vs 0.0)

This isolates the effect of stimulation amplitude from initialization effects.

In [ ]:
# --- Generate control dataset with SAME REST INIT for both conditions ---
print("="*70)
print("GENERATING CONTROL DATASET (STIM_AMP = 0, REST INIT FOR BOTH)")
print("="*70)
print("\nFor each subject with trained model:")
print("  1. Simulate REST sessions (no stimulation, from REST init)")
print("  2. Simulate STIM sessions (STIM_AMP=0, FROM SAME REST INIT)")
print("     → Both use REST initialization for true amplitude-only comparison")
print()

STIM_AMP_CONTROL = 0.0  # Control: zero amplitude

dataset_sim_control = {}
n_sim_rest_control = 0
n_sim_stim_control = 0
sim_errors_control = []

for sub_id in sorted(subjects_with_models_and_data):
    model = trained_models[sub_id]
    sub_data_emp = dataset_emp[sub_id]

    dataset_sim_control[sub_id] = {"task-rest": {}, "task-stim": {}}

    # ----- FIRST PASS: COLLECT REST DATA FOR INITIALIZATION -----
    rest_init_windows = []  # Store (init_window, n_steps) pairs from rest sessions
    
    if "task-rest" in sub_data_emp:
        rest_count = 0
        for run_idx, run_emp in sub_data_emp["task-rest"].items():
            ts_emp = run_emp.get("time series", None)

            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            try:
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                init_window = ts_proc[:using_steps].copy()
                n_steps = ts_proc.shape[0]
                rest_init_windows.append((init_window, n_steps))

                # Simulate REST (from REST init, no stimulation)
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=None, target_idx=None, W=None)

                dataset_sim_control[sub_id]["task-rest"][int(run_idx)] = {
                    "time series": sim_ts,
                    "metadata": {"simulated": True, "tr": float(tr_rest), "n_vols": n_steps}
                }
                rest_count += 1
                n_sim_rest_control += 1
            except Exception as e:
                sim_errors_control.append(f"{sub_id} rest run {run_idx}: {str(e)[:60]}")

        if rest_count > 0:
            print(f"  {sub_id}: Simulated {rest_count} rest sessions (from REST init)")

    # ----- SECOND PASS: SIMULATE STIM WITH CONTROL AMPLITUDE (STIM_AMP=0) & REST INIT -----
    if "task-stim" in sub_data_emp and rest_init_windows:
        stim_count = 0
        for run_idx, run_emp in sub_data_emp["task-stim"].items():
            ts_emp = run_emp.get("time series", None)
            target_vec = run_emp.get("target", None)
            stim_time_df = run_emp.get("stim time", None)

            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            target_idx = safe_target_idx(target_vec)
            if target_idx is None:
                continue

            try:
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                # Extract stimulus timing
                stim_steps_model = []
                if stim_time_df is not None and len(stim_time_df) > 0:
                    onset_col = get_onset_column(stim_time_df)
                    if onset_col is not None:
                        onsets_s = stim_time_df[onset_col].values
                        onsets_s = onsets_s[onsets_s >= remove_initial_trs * tr_stim]
                        onsets_s = onsets_s - remove_initial_trs * tr_stim
                        stim_steps_model = map_onsets_to_steps(onsets_s, tr_model=tr_stim)

                if len(stim_steps_model) == 0:
                    continue

                # USE REST INITIALIZATION (NOT stim initialization)
                # Pick a matching-length rest window for initialization
                n_steps = ts_proc.shape[0]
                matching_rest = [w for w, ns in rest_init_windows if ns >= n_steps]
                
                if not matching_rest:
                    # Use the longest available rest window
                    matching_rest = [max(rest_init_windows, key=lambda x: x[1])[0]]
                
                init_window = matching_rest[0]  # Use first matching REST init

                # CONTROL: Apply STIM_AMP=0 stimulation from REST initialization
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=stim_steps_model, target_idx=target_idx, W=W,
                                    stim_amplitude=STIM_AMP_CONTROL)

                dataset_sim_control[sub_id]["task-stim"][int(run_idx)] = {
                    "time series": sim_ts,
                    "target": target_vec,
                    "stim time": stim_time_df,
                    "metadata": {
                        "simulated": True,
                        "tr": float(tr_stim),
                        "target_idx": int(target_idx),
                        "n_stim_pulses": len(stim_steps_model),
                        "n_vols": n_steps,
                        "stim_amp_applied": 0.0,  # Mark as control
                        "init_type": "REST",  # Mark that REST init was used
                    }
                }
                stim_count += 1
                n_sim_stim_control += 1
            except Exception as e:
                sim_errors_control.append(f"{sub_id} stim run {run_idx}: {str(e)[:60]}")

        if stim_count > 0:
            print(f"  {sub_id}: Simulated {stim_count} stim sessions (STIM_AMP=0, REST init)")

print("\n" + "="*70)
print(f"✓ [CONTROL] Generated {n_sim_rest_control} rest sessions and {n_sim_stim_control} stim sessions (STIM_AMP=0)")
print(f"✓ Both REST and STIM (control) use REST empirical initialization")
print(f"✓ Simulated {len(dataset_sim_control)} subjects total")

if sim_errors_control:
    print(f"\n⚠ Encountered {len(sim_errors_control)} errors during control simulation:")
    for err in sim_errors_control[:3]:
        print(f"  {err}")
    if len(sim_errors_control) > 3:
        print(f"  ... and {len(sim_errors_control)-3} more")


## Step 8: Save Simulated Datasets

In [ ]:
# --- Save both datasets ---
print("\nSaving results to preprocessed_subjects_tms_fmri...")

# Save normal dataset
with open(out_pkl, "wb") as f:
    pickle.dump(dataset_sim, f)
print(f"✓ Saved normal simulated dataset to {out_pkl}")

# Save control dataset
out_pkl_control = os.path.join(out_dir, "dataset_simulated_control_REST_INIT_only.pkl")
with open(out_pkl_control, "wb") as f:
    pickle.dump(dataset_sim_control, f)
print(f"✓ Saved control simulated dataset to {out_pkl_control}")

# Create summaries
summary = {
    "metadata": {
        "date": pd.Timestamp.now().isoformat(),
        "model_type": "stimulus-agnostic MLP",
        "n_subjects_simulated": len(dataset_sim),
        "n_rest_sessions": n_sim_rest,
        "n_stim_sessions": n_sim_stim,
    },
    "parameters": {
        "ROI_num": ROI_num,
        "using_steps": using_steps,
        "tr_rest": tr_rest,
        "tr_stim": tr_stim,
        "remove_initial_trs": remove_initial_trs,
        "filter_freq": [low_hz, high_hz],
        "BURN_IN": BURN_IN,
        "NOISE_SIGMA": NOISE_SIGMA,
        "STIM_AMP": STIM_AMP,
        "RHO_MM": RHO_MM,
    },
    "subjects": list(dataset_sim.keys()),
}

with open(results_json, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved normal summary to {results_json}")

summary_control = {
    "metadata": {
        "date": pd.Timestamp.now().isoformat(),
        "model_type": "stimulus-agnostic MLP",
        "condition": "CONTROL: STIM_AMP=0, REST INIT for both conditions",
        "n_subjects_simulated": len(dataset_sim_control),
        "n_rest_sessions": n_sim_rest_control,
        "n_stim_sessions": n_sim_stim_control,
    },
    "parameters": {
        "ROI_num": ROI_num,
        "using_steps": using_steps,
        "tr_rest": tr_rest,
        "tr_stim": tr_stim,
        "remove_initial_trs": remove_initial_trs,
        "filter_freq": [low_hz, high_hz],
        "BURN_IN": BURN_IN,
        "NOISE_SIGMA": NOISE_SIGMA,
        "STIM_AMP": STIM_AMP_CONTROL,
        "RHO_MM": RHO_MM,
        "initialization": "REST empirical data for both REST and STIM (control)"
    },
    "subjects": list(dataset_sim_control.keys()),
}

results_json_control = os.path.join(out_dir, "simulation_summary_control_REST_INIT.json")
with open(results_json_control, "w") as f:
    json.dump(summary_control, f, indent=2)
print(f"✓ Saved control summary to {results_json_control}")

print(f"\n{'='*70}")
print(f"NEW SIMULATION PARAMETERS (v2):")
print(f"  BURN_IN: {BURN_IN}")
print(f"  NOISE_SIGMA: {NOISE_SIGMA}")
print(f"  STIM_AMP: {STIM_AMP}")
print(f"  RHO_MM: {RHO_MM}")
print(f"\nKEY DIFFERENCE IN CONTROL:")
print(f"  ✅ Both REST and STIM (amplitude=0) use REST initialization")
print(f"  ✅ True amplitude-only comparison (same brain state, different amplitude)")
print(f"{'='*70}")